**OCR chữ Hán: GitHub giữ code, Colab chạy GPU, Drive giữ dữ liệu**

Chọn Runtime > Change runtime type > GPU. Đưa PDF vào
`My Drive/NLP_midterm_zh/source/`. Chạy các ô từ trên xuống.
Mặc định thử trang 3–5; đổi `PAGES` thành `all` sau khi kiểm tra.
Notebook không chứa bản sao code xử lý; mọi thay đổi nằm trong repo.


In [22]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/nguyentrongvanviet/OCR-chinese-vietnamese-Buddhism.git' #@param {type:"string"}
REF = "main" #@param {type:"string"}
REPO_DIR = Path("/content/han-ocr-colab")
if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Điền URL GitHub của repo trước.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.run(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
                            text=True, capture_output=True, check=True).stdout.strip()
    if remote.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise ValueError("Thư mục clone hiện tại thuộc repo khác.")
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           text=True, capture_output=True, check=True).stdout.strip()
    if dirty:
        raise RuntimeError("Bản clone có thay đổi cục bộ; lưu thay đổi trước khi cập nhật.")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        text=True, capture_output=True, check=True).stdout.strip()
print("Code commit:", commit)


Code commit: 9e65fe78ea7204a65040f61a6e63c2698f83bbb6


In [23]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/NLP_midterm_zh" #@param {type:"string"}
OUTPUT_NAME = "ocr_zh_v01_full" #@param {type:"string"}
PAGES = "all" #@param {type:"string"}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
subprocess.run([sys.executable, str(REPO_DIR / "setup_colab.py")], check=True)


CompletedProcess(args=['/usr/bin/python3', '/content/han-ocr-colab/setup_colab.py'], returncode=0)

In [ ]:
import subprocess
command = ["/content/han-ocr-env/bin/python", "-u", str(REPO_DIR / "run_ocr.py"), "--drive-dir", DRIVE_DIR, "--output-name", OUTPUT_NAME, "--pages", PAGES]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    result_code = process.wait()
if result_code:
    raise RuntimeError(f"OCR failed with code {result_code}; see log above")


PDF nguồn: /content/drive/MyDrive/NLP_midterm_zh/source/Ấn Quang Pháp Sư Văn Sao - Zh - 4.pdf
Kết quả: /content/drive/MyDrive/NLP_midterm_zh/ocr_zh_v01_full
PDF: 214 trang. Đã chọn: 214. Cần chạy: 214.
Creating model: ('PP-DocLayoutV3', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayoutV3`.
/content/han-ocr-env/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PaddleOCR-VL-1.6-0.9B', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PaddleOCR-VL-1.6`.
Bucketed engine_config has no entry for resolved engine 'pa

In [ ]:
import subprocess
from pathlib import Path
def run_visible(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout: print(line, end="", flush=True)
        code=p.wait()
    if code: raise RuntimeError(f"Command failed: {code}")
ENV_PY="/content/han-ocr-env/bin/python"
run_visible([ENV_PY,"-m","pip","install","dagshub==0.7.2","boto3>=1.34,<2"])
run_visible([ENV_PY,"-m","pip","check"])

In [ ]:
DAGSHUB_REPO="nguyentrongvanviet/OCR_chinese_to_vietnamese_for_NLP"
auth_code = "import dagshub.auth; from dagshub import get_repo_bucket_client; dagshub.auth.get_token(); client=get_repo_bucket_client('" + DAGSHUB_REPO + "'); client.list_objects_v2(Bucket='" + DAGSHUB_REPO.split("/")[-1] + "',MaxKeys=1); print('DagsHub bucket access OK')"
run_visible([ENV_PY,"-u","-c",auth_code])

In [ ]:
from pathlib import Path
import subprocess,json,os
old_process=globals().get("process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try: old_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        old_process.kill()
        old_process.wait()
REPO_DIR=Path("/content/han-ocr-colab")
assert not subprocess.check_output(["git","-C",str(REPO_DIR),"status","--porcelain"],text=True).strip()
subprocess.run(["git","-C",str(REPO_DIR),"fetch","origin","main"],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"checkout","--detach","FETCH_HEAD"],check=True)
assert subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip().startswith("9e65fe7")
ENV_PY="/content/han-ocr-env/bin/python"
PDF_PATH=REPO_DIR/"assets/pdf/an_quang_van_sao_zh_4.pdf"
OUTPUT_DIR=Path("/content/drive/MyDrive/NLP_midterm_zh/ocr_zh_v02")
PAGES="3-8"
command=[ENV_PY,"-u",str(REPO_DIR/"run_ocr.py"),"--pdf",str(PDF_PATH),"--output",str(OUTPUT_DIR),"--pages",PAGES]
child_env=dict(os.environ,FLAGS_allocator_strategy="naive_best_fit")
print("Suy luận tuần tự, allocator:",child_env["FLAGS_allocator_strategy"],flush=True)
with subprocess.Popen(command,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    try:
        for line in process.stdout: print(line,end="",flush=True)
        result_code=process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
if result_code: raise RuntimeError(f"OCR kết thúc với mã {result_code}")
latest=json.loads((OUTPUT_DIR/"latest_export.json").read_text())
bundle=Path(latest["path"])
manifest=json.loads((bundle/"manifest.json").read_text())
assert manifest["done_pages"]==list(range(3,9)),manifest["missing_pages"]
print("ĐÃ TRÍCH XUẤT ĐỦ 6 TRANG NỘI DUNG: 3, 4, 5, 6, 7, 8",flush=True)
print("ZIP:",bundle/"bundle.zip",flush=True)

In [ ]:
from pathlib import Path
import base64,html,io,json,shutil
from PIL import Image
from IPython.display import HTML,display
OUTPUT_DIR=Path("/content/drive/MyDrive/NLP_midterm_zh/ocr_zh_v02")
bundle=Path(json.loads((OUTPUT_DIR/"latest_export.json").read_text())["path"])
rows=[]
sections=[]
for n in range(3,9):
    folder=OUTPUT_DIR/"pages"/f"page_{n:04d}"
    text=(folder/"text.txt").read_text(encoding="utf-8")
    assert json.loads((folder/"status.json").read_text())["status"]=="done"
    rows.append(f"<tr><td>Trang PDF {n}</td><td>{len(text)} ký tự</td><td>Đã lấy được text</td></tr>")
    with Image.open(folder/"source.png") as im:
        im=im.convert("RGB")
        im.thumbnail((550,900))
        b=io.BytesIO()
        im.save(b,format="JPEG",quality=85)
    encoded=base64.b64encode(b.getvalue()).decode()
    sections.append(f"<details open><summary>Trang PDF {n} · {len(text)} ký tự</summary><div class='pair'><img src='data:image/jpeg;base64,{encoded}' alt='Trang {n}'><pre>{html.escape(text)}</pre></div></details>")
review="<!doctype html><html lang='vi'><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>OCR 6 trang PDF 3–8</title><style>body{font:16px system-ui;max-width:1200px;margin:24px auto;padding:0 18px;color:#192329;background:#f8fafc}h1{font-size:26px}table{border-collapse:collapse;width:100%;background:white;margin:20px 0}td,th{border:1px solid #ccd5df;padding:10px;text-align:left}details{background:white;border:1px solid #ccd5df;border-radius:10px;padding:14px;margin:18px 0}summary{font-size:19px;font-weight:700;cursor:pointer}.pair{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:20px;margin-top:15px}img{width:100%;align-self:start}pre{font-family:serif;white-space:pre-wrap;overflow-wrap:anywhere;line-height:1.9;margin:0}</style><h1>Đã lấy đủ 6 trang PDF: 3–8</h1><p>OCR thô, chưa hiệu đính chữ lem. Mỗi trang gồm ảnh gốc và văn bản nhận dạng để đối chiếu.</p><table><tr><th>Trang</th><th>Văn bản</th><th>Trạng thái</th></tr>"+"".join(rows)+"</table>"+"".join(sections)+"</html>"
review_path=OUTPUT_DIR/"review_0003_0008.html"
review_path.write_text(review,encoding="utf-8")
shutil.copy2(review_path,"/content/ocr_6_pages_review.html")
shutil.copy2(bundle/"han.txt","/content/han_6_pages.txt")
shutil.copy2(bundle/"bundle.zip","/content/ocr_6_pages.zip")
display(HTML(review))

In [ ]:
import base64, html, io, json, shutil
from PIL import Image
from IPython.display import HTML, display
rows=[]
sections=[]
for number in range(3,9):
    folder=OUTPUT_DIR/"pages"/f"page_{number:04d}"
    text=(folder/"text.txt").read_text(encoding="utf-8")
    status=json.loads((folder/"status.json").read_text())
    assert status["status"]=="done"
    rows.append(f"<tr><td><a href='#page-{number}'>Trang PDF {number}</a></td><td>{len(text)}</td><td>Có kết quả OCR</td></tr>")
    with Image.open(folder/"source.png") as picture:
        picture=picture.convert("RGB")
        picture.thumbnail((550,900))
        buffer=io.BytesIO()
        picture.save(buffer,format="JPEG",quality=85)
    encoded=base64.b64encode(buffer.getvalue()).decode("ascii")
    sections.append(f"<details id='page-{number}' open><summary>Trang PDF {number} · {len(text)} ký tự</summary><div class='pair'><img alt='Ảnh gốc trang {number}' src='data:image/jpeg;base64,{encoded}'><pre>{html.escape(text) if text.strip() else '[OCR RỖNG — CẦN XEM ẢNH]'}</pre></div></details>")
review="<!doctype html><html lang='vi'><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>OCR 6 trang nội dung</title><style>body{font:16px system-ui,sans-serif;max-width:1200px;margin:24px auto;padding:0 18px;color:#192329;background:#f8fafc}h1{font-size:26px}table{border-collapse:collapse;width:100%;background:white;margin:20px 0}th,td{border:1px solid #d5dee6;padding:10px;text-align:left}details{margin:18px 0;border:1px solid #cad5df;border-radius:10px;background:white;padding:14px}summary{cursor:pointer;font-weight:700;font-size:19px}.pair{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:20px;margin-top:15px}.pair img{width:100%;height:auto;align-self:start}pre{white-space:pre-wrap;overflow-wrap:anywhere;line-height:1.9;font-family:serif;margin:0}</style><h1>Đã trích xuất 6 trang PDF: 3–8</h1><p>OCR thô, giữ nguyên kết quả mô hình. Chữ lem có thể bị nhận dạng sai hoặc đọc thêm. Ảnh bên cạnh dùng để đối chiếu.</p><table><tr><th>Trang</th><th>Số ký tự</th><th>Kết quả</th></tr>"+"".join(rows)+"</table>"+"".join(sections)+"</html>"
review_path=OUTPUT_DIR/"review_0003_0008.html"
review_path.write_text(review,encoding="utf-8")
shutil.copy2(review_path,"/content/ocr_6_pages_review.html")
shutil.copy2(bundle/"han.txt","/content/han_6_pages.txt")
shutil.copy2(bundle/"bundle.zip","/content/ocr_6_pages.zip")
print("ĐỦ 6/6 TRANG. HTML đối chiếu:",review_path)
print("TXT tổng hợp:",bundle/"han.txt")
print("ZIP gồm TXT từng trang + JSON gốc:",bundle/"bundle.zip")
display(HTML(review))
import zipfile
from google.colab import files
package=Path("/content/ocr_6_pages_results.zip")
with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED) as archive:
    archive.write("/content/ocr_6_pages_review.html","review.html")
    archive.write("/content/han_6_pages.txt","han_6_pages.txt")
    archive.write("/content/ocr_6_pages.zip","ocr_pages_3_8.zip")
print("Tải kết quả 6 trang:",package,package.stat().st_size,"bytes")
files.download(str(package))

In [ ]:
import json
from IPython.display import display
from PIL import Image

output = Path(DRIVE_DIR) / OUTPUT_NAME
print((output / "NOTE.txt").read_text(encoding="utf-8"))
summary = json.loads((output / "summary.json").read_text(encoding="utf-8"))
print("Đã xử lý:", len(summary["done_pages"]), "/", summary["total_pdf_pages"])
PREVIEW_PAGE = 8 #@param {type:"integer"}
page = output / "pages" / f"page_{PREVIEW_PAGE:04d}"
if (page / "source.png").exists():
    with Image.open(page / "source.png") as image:
        image.thumbnail((600, 1000))
        display(image.copy())
if (page / "text.txt").exists():
    print((page / "text.txt").read_text(encoding="utf-8"))


**Tiếp tục chạy**

Sau khi kiểm tra pilot, đổi `PAGES` thành `all` hoặc một khoảng như `6-30`, rồi chạy lại ô OCR.
Khi runtime mới: chạy lại các ô từ đầu; checkpoint hợp lệ trên Drive được bỏ qua.
Clone, cài thư viện và tải model vẫn có thể cần thực hiện lại trong runtime mới.

`REF=main` lấy code mới tại thời điểm chạy ô clone. Có thể điền tag hoặc commit cụ thể để tái lập;
không cập nhật code giữa một lượt OCR. Thay đổi worker/mô hình có thể yêu cầu `OUTPUT_NAME` mới.

`han_ocr_full.txt` chỉ có khi mọi trang đã xử lý; văn bản vẫn là OCR thô.
Chưa khử lem theo cặp hoặc xác nhận độ chính xác OCR. Xem README trong repo để biết giới hạn.
